## Data Cleaning: NY Workers' Compensation Claims 2000–2025

In [5]:
import matplotlib as mpl
import matplotlib.pyplot as plt 
%matplotlib inline
import seaborn as sns
import pandas as pd
sns.set_style("darkgrid")
mpl.rcParams['figure.figsize'] = (20,5)

WC_data_raw = pd.read_csv('Assembled_Workers__Compensation_Claims___Beginning_2000 (version 1).xlsb.csv')

### DATA CLEANING

In [3]:
# Create a copy of the raw data
WC_data_clean = WC_data_raw.copy()

# 1. Drop duplicates
WC_data_clean.drop_duplicates(inplace=True)

# 2. Keep only relevant columns
cols_to_keep = [
    'Accident Date', 'Hearing Count', 'Age at Injury', 'Gender', 'County of Injury', 
    'District Name', 'Zip Code', 'Industry Code', 'Industry Code Description',
    'Claim Injury Type', 'Claim Type', 'Attorney/Representative', 
    'Alternative Dispute Resolution', 'Current Claim Status', 'Highest Process'
    
]
WC_data_clean = WC_data_clean[cols_to_keep]

# 3. Flag missing Accident Dates 
WC_data_clean['Date Unknown'] = WC_data_clean['Accident Date'].isna()

# 4. Fill missing Industry Code values
WC_data_clean['Industry Code'] = WC_data_clean['Industry Code'].fillna(0)
WC_data_clean['Industry Code Description'] = WC_data_clean['Industry Code Description'].fillna('Unknown')

# 5. Convert data types
WC_data_clean['Accident Date'] = pd.to_datetime(WC_data_clean['Accident Date'])
WC_data_clean['Industry Code'] = pd.to_numeric(WC_data_clean['Industry Code'], errors='coerce').fillna(0).astype(int)

# 6. Create a column to flag unknown ages
WC_data_clean['Age Unknown'] = WC_data_clean['Age at Injury'] == 0

# 7. Filter to match analysis date range and realistic ages
WC_data_clean = WC_data_clean[(WC_data_clean['Accident Date'].dt.year >= 2000) & 
                               (WC_data_clean['Accident Date'].dt.year <= 2025) |
                               (WC_data_clean['Date Unknown'] == True)]
WC_data_clean = WC_data_clean[WC_data_clean['Age at Injury'] <= 100]

# 8. Check for leading/trailing spaces in text columns 
for col in WC_data_clean.select_dtypes(include='object').columns:
    has_spaces = WC_data_clean[col].str.strip().ne(WC_data_clean[col]).sum()
    if has_spaces > 0:
        print(f"{col}: {has_spaces} values with leading/trailing spaces")

# 9. Create Age Group column
WC_data_clean['Age Group'] = WC_data_clean['Age at Injury'].apply(
    lambda x: 'Unknown' if x == 0 else
              'Teens (13-19)' if x < 20 else
              '20s (20-29)' if x < 30 else
              '30s (30-39)' if x < 40 else
              '40s (40-49)' if x < 50 else
              '50s (50-59)' if x < 60 else
              '60s (60-69)' if x < 70 else
              '70+ (70+)'
)

# 10. Fix Zip Codes with missing leading zeros
WC_data_clean['Zip Code'] = WC_data_clean['Zip Code'].str.zfill(5)

# 11. Flag invalid Zip Codes
WC_data_clean['Zip Unknown'] = (
    (WC_data_clean['Zip Code'] == '00000') |
    (~WC_data_clean['Zip Code'].str.match(r'^\d{5}$'))
)

# 12. Fix double spaces in Claim Type
WC_data_clean['Claim Type'] = WC_data_clean['Claim Type'].str.replace('  ', ' ')


### FINAL CHECK

In [4]:
print("Raw shape:", WC_data_raw.shape)
print("Cleaned shape:", WC_data_clean.shape)

print("\nMissing values in cleaned data:")
print(WC_data_clean.isnull().sum())

Raw shape: (1048575, 54)
Cleaned shape: (1019966, 19)

Missing values in cleaned data:
Accident Date                     10147
Hearing Count                         0
Age at Injury                         0
Gender                                0
County of Injury                      0
District Name                         0
Zip Code                              0
Industry Code                         0
Industry Code Description             0
Claim Injury Type                     0
Claim Type                            0
Attorney/Representative               0
Alternative Dispute Resolution        0
Current Claim Status                  0
Highest Process                       0
Date Unknown                          0
Age Unknown                           0
Age Group                             0
Zip Unknown                           0
dtype: int64
